In [1]:
import torch

# Check for NVIDIA GPU (CUDA)
print("CUDA Available:", torch.cuda.is_available())

# Check for Apple Silicon Mac GPU (MPS)
print("MPS Available:", torch.backends.mps.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

CUDA Available: True
MPS Available: False
Using device: cuda


In [2]:
import pandas as pd

family_df = pd.read_csv('protein_families.csv')

i = 0
name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]

In [3]:
from Bio import SeqIO
from modules.sequences import split_train_val_test

# read the fasta files
recs = [(r.id, str(r.seq)) for r in SeqIO.parse(f"sequences/{accession}.fasta", "fasta")]

# separate to train, validation, and test set
train_seqs, val_seqs, test_seqs = split_train_val_test(recs)

In [4]:
from modules.model import (
    init_transition_mat,
    init_emission_mat
)

# init transition and emmission matrices
trans_matdf = init_transition_mat(match_len)
emit_matdf = init_emission_mat(match_len, seed=2)

# encode the symbols into index for the hmmlearn
symbols = list(emit_matdf.index)            # ^ A C D ... Y $  (taken from the CSV row order)
sym_idx = {s: i for i, s in enumerate(symbols)}
BEGIN = sym_idx["^"]                         # dummy begin symbol, emitted only by M0
END = sym_idx["$"]    

In [5]:
from modules.qmodel import (
    build_hqmm,
    convert_to_torch
)

K = build_hqmm(trans_matdf, emit_matdf)
# K_torch = convert_to_torch(K)

In [6]:
from modules.qmodel import (
    compress_phmm_kraus
)

n = 2 * match_len + 1
s = 21
W = 1

# Your existing PHMM-to-HQMM conversion
# must provide p_phmm and K_phmm, with W=n.

K_initial = compress_phmm_kraus(
    K,
    n=n,
    W=n-10,
    device=device,
    seed=2
)

# Initial density matrix
p = torch.zeros(K[0, 1].shape, dtype=torch.complex64)
p[0, 0] = 1.0

In [7]:

import torch
import pandas as pd
from pathlib import Path

from modules.qmodel import (
    build_hqmm,
    compress_phmm_kraus,
    convert_to_torch
)
from modules.qtrain import train_hqmm
from modules.qevaluation import description_accuracy_hqmm
from modules.model import (
    init_transition_mat,
    init_emission_mat
)

# ==========================================
# 1. MODEL CONFIGURATION
# ==========================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

n = match_len * 2 + 1
s = 21  # 20 amino acids + $

# Initialize PHMM transition and emission matrices
trans_matdf = init_transition_mat(match_len)
emit_matdf = init_emission_mat(match_len, seed=2)

symbols = list(emit_matdf.index)
sym_idx = {
    symbol: i
    for i, symbol in enumerate(symbols)
}

BEGIN = sym_idx["^"]
END = sym_idx["$"]

assert BEGIN == 0
assert sorted(
    idx for symbol, idx in sym_idx.items()
    if symbol != "^"
) == list(range(1, s + 1))

# ==========================================
# 2. ORIGINAL PHMM-DERIVED KRAUS OPERATORS
# ==========================================

# Construct the original W=n Kraus operators
# using the square-root observable-operator method.

K_phmm = build_hqmm(
    trans_matdf,
    emit_matdf
)

K_phmm = convert_to_torch(K_phmm)

# Exclude the start symbol (^)
K_phmm = {
    key: value
    for key, value in K_phmm.items()
    if key[1] != BEGIN
}

print(
    f"Original PHMM-derived HQMM: "
    f"n={n}, W={n}, operators={len(K_phmm)}"
)

# ==========================================
# 3. TRAINING CONFIGURATION
# ==========================================

# Vary the number of Kraus operators
settings = {
    f"w={w}": w
    for w in sorted(set([2, int(n-70), int(n-50), int(n-30)]))
}

# Fixed optimization parameters
tau = 0.1
alpha = 0.8
max_iter = 15
tol = 1e-3
batch_size = 16

results = {}
histories = {}
trained_models = {}

model_dir = Path("saved_models/phqmm")
model_dir.mkdir(parents=True, exist_ok=True)

# ==========================================
# 4. INITIAL DENSITY MATRIX
# ==========================================

# Use the same initial state as the original
# PHMM-derived HQMM.

p = torch.zeros(
    (n, n),
    dtype=torch.complex64,
    device=device
)

p[0, 0] = 1.0

# ==========================================
# 5. TRAINING LOOP
# ==========================================

for label, W in settings.items():

    print(
        f"\n--- {label}: n={n}, W={W}, tau={tau}",
        flush=True
    )

    # --------------------------------------
    # Initialize Kraus operators
    # --------------------------------------

    if W == n:
        # Exact original PHMM-derived initialization
        K_initial = {
            key: value.detach().clone().to(
                device=device,
                dtype=torch.complex64
            )
            for key, value in K_phmm.items()
        }

    else:
        # Compress PHMM-derived operators to W<n
        K_initial = compress_phmm_kraus(
            K_phmm=K_phmm,
            n=n,
            W=W,
            device=device,
            seed=100 + W
        )

        K_initial = {
            key: value.detach().clone().to(
                device=device,
                dtype=torch.complex64
            )
            for key, value in K_initial.items()
        }

    # --------------------------------------
    # Verify Kraus completeness
    # --------------------------------------

    with torch.no_grad():
        S = torch.zeros(
            (n, n),
            dtype=p.dtype,
            device=device
        )

        for A in K_initial.values():
            S += A.conj().T @ A

        I = torch.eye(
            n,
            dtype=p.dtype,
            device=device
        )

        completeness_error = (
            torch.linalg.matrix_norm(S - I).item()
        )

        print(
            f"Completeness error: "
            f"{completeness_error:.6e}",
            flush=True
        )

        if not torch.allclose(
            S, I, atol=1e-4, rtol=1e-4
        ):
            raise ValueError(
                f"Kraus completeness failed for W={W}"
            )

    # --------------------------------------
    # Evaluate untrained model
    # --------------------------------------

    with torch.no_grad():
        da0 = description_accuracy_hqmm(
            p,
            K_initial,
            W,
            sym_idx,
            test_seqs
        )

    results[f"{label} (untrained)"] = (
        float(da0.mean()),
        float(da0.std())
    )

    print(
        f"Initial Test DA: "
        f"{results[f'{label} (untrained)'][0]:.4f} "
        f"± {results[f'{label} (untrained)'][1]:.4f}",
        flush=True
    )

    # --------------------------------------
    # Train HQMM using COSM
    # --------------------------------------

    K_trained, hist = train_hqmm(
        p=p,
        K=K_initial,
        W=W,
        sym_idx=sym_idx,
        train_seqs=train_seqs,
        val_seqs=val_seqs,
        test_seqs=test_seqs,
        max_iter=max_iter,
        tol=tol,
        tau=tau,
        alpha=alpha,
        batch_size=batch_size,
        verbose=True,
        device=device
    )

    histories[label] = hist
    trained_models[label] = K_trained

    # --------------------------------------
    # Save checkpoint and history
    # --------------------------------------

    prefix = (
        f"{name}_{accession}_m{match_len}_"
        f"n{n}_w{W}_tau{tau:g}_alpha{alpha:g}"
    )

    model_path = model_dir / (
        f"{prefix}_hqmm.pt"
    )

    history_path = model_dir / (
        f"{prefix}_history.csv"
    )

    checkpoint = {
        "K": {
            key: value.detach().cpu().clone()
            for key, value in K_trained.items()
        },
        "p": p.detach().cpu().clone(),
        "n": n,
        "s": s,
        "W": W,
        "initialization": (
            "phmm_exact"
            if W == n
            else "phmm_compressed"
        ),
        "tau": tau,
        "alpha": alpha,
        "batch_size": batch_size,
        "max_iter": max_iter,
        "tol": tol,
        "epochs_completed": len(hist) - 1,
    }

    torch.save(checkpoint, model_path)
    hist.to_csv(history_path, index=False)

    print(f"Model saved: {model_path}", flush=True)
    print(f"History saved: {history_path}", flush=True)

    # --------------------------------------
    # Evaluate trained model
    # --------------------------------------

    K_ref = next(iter(K_trained.values()))

    p_eval = p.to(
        device=K_ref.device,
        dtype=K_ref.dtype
    )

    K_eval = {
        key: value.to(
            device=K_ref.device,
            dtype=K_ref.dtype
        )
        for key, value in K_trained.items()
    }

    with torch.no_grad():
        da = description_accuracy_hqmm(
            p_eval,
            K_eval,
            W,
            sym_idx,
            test_seqs
        )

    results[label] = (
        float(da.mean()),
        float(da.std())
    )

    print(
        f"Trained Test DA: "
        f"{results[label][0]:.4f} "
        f"± {results[label][1]:.4f}",
        flush=True
    )

# ==========================================
# 6. FINAL SUMMARY
# ==========================================

summary = []

for label, W in settings.items():

    initial_mean, initial_std = results[
        f"{label} (untrained)"
    ]

    trained_mean, trained_std = results[label]

    summary.append({
        "n": n,
        "W": W,
        "Initial DA Mean": initial_mean,
        "Initial DA Std": initial_std,
        "Trained DA Mean": trained_mean,
        "Trained DA Std": trained_std,
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

summary_path = model_dir / (
    f"{name}_{accession}_m{match_len}_"
    f"n{n}_kraus_rank_comparison.csv"
)

summary_df.to_csv(summary_path, index=False)

Original PHMM-derived HQMM: n=79, W=79, operators=1659

--- w=2: n=79, W=2, tau=0.1
Completeness error: 9.815134e-07
Initial Test DA: 0.0049 ± 0.0042
Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14606.13  train_da_mean 0.0048  val_da_mean 0.0069
  iter   1  batch   1/5  loss 222.7965  tau 0.100000
  iter   1  batch   2/5  loss 192.3386  tau 0.100000
  iter   1  batch   3/5  loss 183.2448  tau 0.100000
  iter   1  batch   4/5  loss 153.9357  tau 0.100000
  iter   1  batch   5/5  loss 159.8143  tau 0.100000
iter   1  train     5.7s  log-lik -10635.48  train_da_mean 0.2893  val_da_mean 0.2717
  iter   2  batch   1/5  loss 171.4240  tau 0.080000
  iter   2  batch   2/5  loss 126.6018  tau 0.080000
  iter   2  batch   3/5  loss 127.1021  tau 0.080000
  iter   2  batch   4/5  loss 166.6237  tau 0.080000
  iter   2  batch   5/5  loss 154.0374  tau 0.080000
iter   2  train    11.1s  log-lik -10711.97  train_da_mean 0.2883  val_da_mean 0.2581
  iter   3  batch   1

,n,W,Initial DA Mean,Initial DA Std,Trained DA Mean,Trained DA Std
0,79,2,0.004855,0.004220,0.354679,0.091027
1,79,9,0.001221,0.002749,0.346119,0.084786
2,79,29,0.001233,0.002645,0.360256,0.087061
3,79,49,0.001271,0.002669,0.354589,0.082630


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from modules.qplots import (
    plot_da_learning_curve,
    plot_da_bars,
)

from pathlib import Path

try:
    plt.style.use("seaborn-v0_8-darkgrid")
except OSError:
    pass

# Create directory for saved models
results_dir = Path("results/phqmm/w-phqmm")

# Save training histories
for label, h in histories.items():
    h.to_csv(
        f"q_history_{label.split()[0]}.csv",
        index=False
    )

# Save combined training histories
combined_histories = pd.concat(
    [
        h.assign(step_size=label)
        for label, h in histories.items()
    ],
    ignore_index=True,
)

combined_histories.to_csv(
    results_dir / f"{name}_q_training_histories.csv",
    index=False,
)

# Save final description accuracy results
pd.DataFrame(
    results,
    index=["test_da_mean", "test_da_std"]
).T.to_csv(results_dir / f"{name}_q_da_results.csv")

# Optional reference DA values
references = {}

# Test DA vs GPU training time
plot_da_learning_curve(
    histories,
    references,
    x="train_time",
    out=results_dir / f"{name}_q_da_vs_time.png"
)

# Test DA vs epoch
plot_da_learning_curve(
    histories,
    references,
    x="iter",
    out=results_dir / f"{name}_q_da_vs_epoch.png"
)

# Final DA comparison by step size
plot_da_bars(
    results,
    references,
    out=results_dir / f"{name}_q_da_results.png"
)